# 6.5 My own vectors: does Louis say different things in French than in English?

> **Question:** before Benji was born, did Louis talk about different things in French
> than in English?

**My expectation:** in French he shares more love and compliments; his English is more
businesslike and practical.

Only the weeks **before Benji's birth** (`settings.BIRTH_DATE`): after that, Louis writes
almost only in French (68% French, 7% English), so there is hardly any English left to
compare with.

The method, from lessons 6.1–6.4:

1. **One row** = one *unit*: the messages of one session (6.4's time gap), split by
   language, so a session with three English and two French messages gives two units.
2. **Vectors** from a *multilingual* sentence encoder (6.3, 6.4), so a French and an
   English unit that mean the same thing get the same vector.
3. **k-means** on those vectors: groups of units that are about the same thing.
4. **Per week**, compare how French and English units spread over those groups, and check
   the difference with a shuffle that holds the week and the unit length fixed (04.6).

> Needs `uv sync --extra huggingface` once. The first run downloads the encoder (~470 MB).

In [ ]:
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")  # joblib's core count crashes on Windows

import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from goad_toolkit.datatransforms import Pipeline
from goad_toolkit.visualizer import HeatmapPlot, LinePlot, PlotSettings, ProjectionPlot, ScreePlot
from notebooktester import param
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score

from roos_pipeline import settings
from roos_pipeline.settings import Paths
from roos_pipeline.vectors import (
    ContrastBarPlot,
    LanguageDumbbellPlot,
    ReportMapPanels,
    StripQuotedReply,
    TopicMapPanels,
    WeeklyLanguagePanels,
    WeekStart,
    WordCount,
    cluster_contrast,
    cluster_examples,
    cluster_keywords,
    contrast_intervals,
    language_units,
    neighbour_agreement,
    topic_labels,
    topic_term_counts,
    weekly_cluster_shares,
)
from scripts.sessionize import fit_session_threshold

LANGUAGE_NAMES = {"fr": "French", "en": "English"}

## 6.5.1 Louis's messages before Benji

From `2_features.parquet` (the pipeline's stage 2), which already carries `language`
(`py3langid`, en/nl/fr, under 3 characters = "unknown"). `StripQuotedReply` first: a reply
that quotes one of my messages would otherwise put my words in Louis's mouth.

In [ ]:
features = pd.read_parquet(Paths().features)
before_benji = features[(features["author"] == "Louis") & (features["ts_local"] < settings.BIRTH_DATE)]
louis = Pipeline().add(StripQuotedReply).add(WeekStart).apply(before_benji.copy())
louis = louis[louis["message"] != ""]

print(f"{len(louis):,} messages from Louis, {louis['week'].nunique()} weeks, "
      f"{louis['ts_local'].min():%d %b %Y} to {louis['ts_local'].max():%d %b %Y}")
louis["language"].value_counts()

## 6.5.2 What is one row? Sessions, split by language

6.4's lesson: one message ("Top", "Hahaha") is too short to mean anything as a vector. So
messages are merged into sessions first, with the gap threshold *fitted* on Louis's own
gaps (`fit_session_threshold`, the lesson 4 exponential), not guessed.

Sessions are cut on all his messages, Dutch included, so switching language mid-conversation
does not start a new session. Only then is each session split into its French and English
part. Dutch is left out of the comparison.

In [ ]:
threshold = fit_session_threshold(louis["ts_local"], louis["author"])
print(f"session threshold: {threshold:.0f} s ({threshold / 60:.1f} minutes)")

units = Pipeline().add(WordCount).apply(language_units(louis, threshold))
units.groupby("language")[["n_messages", "n_words"]].median()

**The first thing to know:** French units are much shorter than English ones. That is not a
detail. A short text lands in a different part of the vector space than a long one, *whatever
it says*: in a first try, k-means with 2 groups split the units into short and long, not by
meaning. Two things handle that:

- units under `settings.MIN_UNIT_WORDS` words are left out (below);
- the shuffle test in 6.5.6 compares French and English only *within the same length band*.

What the 4-word minimum throws away, most common first:

In [ ]:
short = units[units["n_words"] < settings.MIN_UNIT_WORDS]
for language, group in short.groupby("language"):
    top = group["message"].str.lower().str.strip().value_counts().head(12)
    print(f"{LANGUAGE_NAMES[language]}: {len(group):,} short units dropped, e.g. {top.to_dict()}")

units = units[units["n_words"] >= settings.MIN_UNIT_WORDS].reset_index(drop=True)
units.groupby(["language", "length_band"]).size().unstack()

## 6.5.3 Vectors: why a multilingual encoder

6.4 used `all-MiniLM-L6-v2`. That model was trained on English only. For this question that is
fatal: it would put every French unit in one corner because it is *French*, and k-means would
find the language, not what was said. `settings.EMBED_MODEL` was trained so that a sentence and
its translation get the same vector. Four sentences show the difference:

In [ ]:
pairs = ["je t'aime mon amour", "I love you my love", "à quelle heure part le train ?", "what time does the train leave?"]
for name in ["sentence-transformers/all-MiniLM-L6-v2", settings.EMBED_MODEL]:
    v = SentenceTransformer(name).encode(pairs, normalize_embeddings=True)
    print(f"{name.split('/')[-1]:>40s}:  love fr~en {v[0] @ v[1]:.2f}   train fr~en {v[2] @ v[3]:.2f}   "
          f"love~train (fr) {v[0] @ v[2]:.2f}")

In [ ]:
UNIT_SAMPLE = param(None, test=300)  # notebooktester runs on a small sample
if UNIT_SAMPLE is not None:
    units = units.sample(n=min(UNIT_SAMPLE, len(units)), random_state=settings.SEED).reset_index(drop=True)

encoder = SentenceTransformer(settings.EMBED_MODEL)
vectors = encoder.encode(units["message"].tolist(), batch_size=64, normalize_embeddings=True,
                         show_progress_bar=False)
print(f"{vectors.shape[0]:,} units x {vectors.shape[1]} dimensions")

## 6.5.4 300 dimensions?

The number of dimensions is not a choice you make when embedding: the encoder decides it,
384 here. "300 dimensions" can only mean: *reduce* the 384 to 300 with PCA before k-means.
Whether that is a good idea, the scree plot (6.1) answers.

In [ ]:
pca = PCA().fit(vectors)
cumulative = np.cumsum(pca.explained_variance_ratio_)

scree = PlotSettings(figsize=(8, 4), title="How much of the 384 dimensions do the first n components hold?",
                     xlabel="number of components", ylabel="share of variance")
fig, ax = ScreePlot(scree).plot(explained_variance_ratio=pca.explained_variance_ratio_)
ax.axvline(300, color=settings.GREY, linestyle="--")
ax.set_xticks([1, 50, 100, 150, 200, 250, 300, 384])
ax.lines[0].set_markersize(2)  # 384 components: dots, not beads

for n in [2, 50, 100, 300]:
    if n <= len(cumulative):
        print(f"{n:>4d} components: {cumulative[n - 1]:.1%} of the variance")
print(f"90% needs {np.searchsorted(cumulative, 0.9) + 1} components")

In [ ]:
K_CHECK = 6
full = KMeans(K_CHECK, n_init=10, random_state=settings.SEED).fit_predict(vectors)
reduced = KMeans(K_CHECK, n_init=10, random_state=settings.SEED).fit_predict(
    PCA(n_components=min(300, len(vectors) - 1)).fit_transform(vectors)
)
print(f"k-means on 384 vs. on 300 PCA components, same k: adjusted Rand index "
      f"{adjusted_rand_score(full, reduced):.2f} (1 = the same groups)")

In [ ]:
N_COMPONENTS = [5, 10, 25, 50, 100, 118, 150, 200, 300]
per_n = pd.DataFrame({"components": [n for n in N_COMPONENTS if n <= len(cumulative)]})
per_n["variance"] = cumulative[per_n["components"].to_numpy() - 1]
per_n["rand_index"] = [
    adjusted_rand_score(full, KMeans(K_CHECK, n_init=10, random_state=settings.SEED).fit_predict(
        PCA(n_components=n).fit_transform(vectors)))
    for n in per_n["components"]
]

# yardstick: all 384 dimensions, only another seed; differences smaller than this are k-means' own luck
other_seed = KMeans(K_CHECK, n_init=10, random_state=settings.SEED + 1).fit_predict(vectors)
seed_only = adjusted_rand_score(full, other_seed)
print(f"384 vs. 384 with another seed: adjusted Rand index {seed_only:.2f}")

ari = PlotSettings(figsize=(8, 3.5), title=f"Same groups as on all 384 dimensions? (k = {K_CHECK})",
                   xlabel="number of PCA components", ylabel="adjusted Rand index")
fig, ax = LinePlot(ari).plot(data=per_n, x="components", y="rand_index", marker="o", color=settings.GREY)
ax.axhline(seed_only, color=settings.LIGHT_GREY, linestyle="--")
ax.set_ylim(0, 1.05)
per_n.style.format({"variance": "{:.1%}", "rand_index": "{:.2f}"})

**How many components, then?** *(read the line and the dashed yardstick first: from how many
components on are the groups as good as the same as on all 384? Is that a reason to reduce?)*

*Answer:* **Keuze: alle 384 dimensies houden, geen PCA vóór k-means.**
Ik heb niet gereduceerd, omdat de check liet zien dat het niets verandert aan de groepen en de
data klein genoeg is. Geen extra stap en geen getal dat ik hoef te onderbouwen.

## 6.5.5 k-means: how many groups?

The vectors are normalised (length 1), so k-means' straight-line distance ranks pairs the same
way cosine does (6.3). Silhouette per k, on the vectors themselves, never on a 2D picture (6.1.6).

In [ ]:
K_RANGE = range(2, 11)
silhouettes = pd.DataFrame({
    "k": list(K_RANGE),
    "silhouette": [
        silhouette_score(vectors, KMeans(k, n_init=10, random_state=settings.SEED).fit_predict(vectors), metric="cosine")
        for k in K_RANGE
    ],
})
line = PlotSettings(figsize=(7, 3.5), title="Silhouette per k (1 = sharp groups, 0 = none)", xlabel="k",
                    ylabel="silhouette")
fig, ax = LinePlot(line).plot(data=silhouettes, x="k", y="silhouette", marker="o", color=settings.GREY)
ax.set_ylim(0, None)
silhouettes.round(3).T

The silhouette stays close to 0 for every k: Louis's units do **not** fall into sharp,
separate groups. k-means will still hand out groups, but they are cuts through one continuous
cloud, like dividing a colour gradient into bands. That is fine for this question (*how much of
the French vs. the English sits in each band*), as long as nobody reads the groups as "the six
kinds of thing Louis talks about".

`K` below is a starting point (6), not a decision: choose it yourself, after reading the
examples. Fewer groups are easier to name; more groups can split "love" from "compliments".

In [ ]:
K = 6  # my choice, because: ...
kmeans = KMeans(K, n_init=10, random_state=settings.SEED).fit(vectors)
units["cluster"] = kmeans.labels_

pd.crosstab(units["cluster"], units["language"].map(LANGUAGE_NAMES), margins=True)

### What is each group about?

The units closest to each group's centre, per language: the most typical members, not a
random draw. Below them, `cluster_keywords` gives each group's most *distinctive* words
(frequent in that group, rare in the others), and the first three become its working name.
That is a draft: read the examples, and **overrule a name** in `CLUSTER_NAMES` where the
words miss the point. Check also whether a group means the same thing in French and in
English, or whether it mostly holds one language.

In [ ]:
for language in settings.COMPARE_LANGUAGES:
    mask = (units["language"] == language).to_numpy()
    examples = cluster_examples(units.loc[mask, "message"], vectors[mask], units.loc[mask, "cluster"].to_numpy(),
                                kmeans.cluster_centers_, n=6)
    print(f"===== {LANGUAGE_NAMES[language]} =====")
    for k, texts in examples.items():
        print(f"--- group {k}")
        for text in texts:
            print("   ", text)

In [ ]:
keywords = cluster_keywords(units["message"], units["cluster"].to_numpy())
for k, words in keywords.items():
    print(f"group {k}: {', '.join(words)}")

# A first draft from the keywords; overrule any group by name here, after reading the examples above.
CLUSTER_NAMES: dict[int, str] = {
    0: "Daily talk: sport, sleep, work",
    1: "Travelling",
    2: "Short responses",
    3: "Words of love",
    4: "Making plans",
    5: "House and stuff",
}
names = {k: CLUSTER_NAMES.get(k, " / ".join(keywords[k][:3])) for k in range(K)}

### The cloud, seen from above

A PCA picture of the 384 dimensions, only to *look*: the groups were made in all 384, and the
first two components hold a small share of the variance (see 6.5.4), so overlap here does not
mean overlap in the data.

In [ ]:
coords = PCA(n_components=2).fit_transform(vectors)
panel = PlotSettings(figsize=(12, 5.5), title=f"Louis's units before Benji, PCA ({cumulative[1]:.0%} of the variance)",
                     xlabel="", ylabel="", max_cols=2,
                     subplot_titles=["coloured by k-means group", "coloured by language"])
host = ProjectionPlot(panel)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(ProjectionPlot(panel), axes[0], coordinates=coords,
                  labels=units["cluster"].map(names).to_numpy(), palette="colorblind", s=6,
                  hue_order=[names[k] for k in range(K)])
_ = host.plot_on_axes(ProjectionPlot(panel), axes[1], coordinates=coords,
                  labels=units["language"].map(LANGUAGE_NAMES).to_numpy(),
                  palette={"French": settings.FRENCH_COLOR, "English": settings.ENGLISH_COLOR}, s=6)

### Which groups lie close together, in all 384 dimensions?

The PCA picture shows 10% of the variance. A heat map of how similar the group centres are
(cosine, 1 = same direction) uses all of it: no squashing, only the groups' averages. Two
groups that are close here are neighbouring bands of the same cloud, not separate topics.

In [ ]:
centres = kmeans.cluster_centers_ / np.linalg.norm(kmeans.cluster_centers_, axis=1, keepdims=True)
labels = [names[k] for k in range(K)]
similarity = pd.DataFrame(centres @ centres.T, index=labels, columns=labels)

heat = PlotSettings(figsize=(1.1 * K + 3, 0.9 * K + 1.5), title="How alike are the group centres? (cosine)",
                    xlabel="", ylabel="")
# the diagonal (a group with itself, always 1) is left out, or it sets the colour scale
fig, ax = HeatmapPlot(heat).plot(data=similarity, fmt=".2f", square=True, mask=np.eye(K, dtype=bool),
                                 cbar_kws={"shrink": 0.7})
ax.tick_params(axis="x", rotation=30)
for tick in ax.get_xticklabels():
    tick.set_ha("right")

## 6.5.6 French against English, all weeks together

For each group: the share of Louis's French units in it, and the share of his English units.
The p-value comes from shuffling the French/English label 10,000 times **within each week and
length band** (`cluster_contrast`): a week keeps its own mix, a length band keeps its own, so
what is left is the difference that week and length cannot explain (`beyond_strata`).

In [ ]:
N_SHUFFLE = param(settings.N_SHUFFLE, test=200)
contrast = cluster_contrast(units, n_shuffle=N_SHUFFLE)

dumbbell = PlotSettings(figsize=(9, 0.6 * K + 1.5),
                        title="Which groups does Louis use more in French, which more in English?",
                        xlabel="share of his units in that language (%)", ylabel="")
fig, ax = LanguageDumbbellPlot(dumbbell).plot(contrast=contrast, names=names,
                                              pair_labels={"fr": "French", "en": "English"})
n_lang = units["language"].map(LANGUAGE_NAMES).value_counts()
fig.text(0.01, -0.02, f"Units of {settings.MIN_UNIT_WORDS}+ words before Benji's birth: "
         f"{n_lang.get('French', 0):,} French, {n_lang.get('English', 0):,} English, "
         f"{units['week'].nunique()} weeks. p: shuffle within week x length band, {N_SHUFFLE:,}x.",
         fontsize=8, color="dimgray")

contrast.rename(index=names).style.format({
    "share_fr": "{:.1%}", "share_en": "{:.1%}", "difference": "{:+.1%}", "shuffle_mean": "{:+.1%}",
    "beyond_strata": "{:+.1%}", "p": "{:.4f}",
})

### The difference itself, and how stable it is

The dumbbell shows both shares; this shows only their difference, in percentage points, so the
groups can be sorted from "most French" to "most English". Two checks sit on each bar
(`contrast_intervals`):

- **the line:** a 95% interval from resampling *weeks*, because one busy week fills one group;
- **the dots:** the difference in odd and in even months apart. Both on the bar's side of 0:
  the difference holds in each half of the data. On opposite sides: it does not.

In [ ]:
N_BOOT = param(2000, test=100)
intervals = contrast_intervals(units, n_boot=N_BOOT)

bars = PlotSettings(figsize=(9, 0.55 * K + 1.8), title="Which groups does Louis use more in French, which more in English?",
                    xlabel="French share − English share (percentage points)", ylabel="")
fig, ax = ContrastBarPlot(bars).plot(intervals=intervals, names=names, pair_labels={"fr": "French", "en": "English"})
fig.text(0.01, -0.02, f"Line: 95% interval, {N_BOOT:,} resamples of the {units['week'].nunique()} weeks. "
         "Dots: the same difference in odd and in even months only.", fontsize=8, color="dimgray")

intervals.rename(index=names).round(1)

## 6.5.7 Week by week

The same comparison per week: one panel per group, the share of that week's French units in
it against the share of that week's English units. A week counts for a language only with at
least `settings.MIN_UNITS_PER_WEEK` units in that language; otherwise one unit makes it 0% or 100%.
Dots are the real weeks, the line a centred 4-week average. Dashed lines: the positive test
(Aug 2024) and the move (Nov 2024).

In [ ]:
weekly = weekly_cluster_shares(units)
weeks_per_language = weekly.groupby("language")["week"].nunique()
print(f"weeks with enough units: French {weeks_per_language.get('fr', 0)}, English {weeks_per_language.get('en', 0)}")

panels = PlotSettings(figsize=(13, 2.6 * ((K + 1) // 2)), title="Share of the week's units, French vs. English",
                      xlabel="", ylabel="%", max_cols=2, sharex=True,
                      subplot_titles=[names[k] for k in range(K)])
fig, axes = WeeklyLanguagePanels(panels).plot(n_plots=K, weekly=weekly, clusters=list(range(K)),
                                              pair_labels={"fr": "French", "en": "English"})
for ax in axes:
    for day in [settings.TEST_DATE, settings.MOVE_DATE]:
        ax.axvline(day, color=settings.LIGHT_GREY, linestyle="--", linewidth=1)

## What I found

*(in my own words, after naming the groups)*

1. **Love and compliments:** which group(s) are they? Is Louis's French share there higher than
   his English share, and does it survive the shuffle (`beyond_strata`, `p`)?
2. **Practical / businesslike:** which group(s)? Higher in English?
3. **Week by week:** is the difference there in most weeks, or does a handful of weeks carry it?
   Does it change after the test or the move (dashed lines)?
4. **Did my expectation hold?** What surprised me?

## Things to check before I believe it

- **Language labels.** `py3langid` calls "hahaha" French and "ouiiii" English. How often is a
  unit's language wrong, or mixed (a French unit with English words in it)? Read 20 at random.
- **Does the encoder still "see" the language?** The PCA picture coloured by language shows it.
  If a group is almost all one language, is that because of *what* was said or *that* it was French?
- **k.** Does the answer to 1 and 2 stay the same with `K` one lower and one higher?
- **The 4-word minimum.** Rerun with `settings.MIN_UNIT_WORDS = 2`: does "bonne nuit 😘" change
  the love result?
- **n = 1 person.** Whatever this shows, it is about Louis in this chat, not about French and English.

## 6.5.8 How hard is the conclusion?

**Check 1: length.** The bar chart (6.5.6) shows the raw difference and does *not* hold length
fixed, while French units are shorter. `beyond_strata` from the shuffle test does: it is what is
left once week and length band are the same for both languages. How much of each difference
survives that?

In [ ]:
held = pd.DataFrame({
    "raw (pts)": intervals["difference"],
    "week & length held (pts)": contrast["beyond_strata"] * 100,
    "p": contrast["p"],
})
held["left over"] = held["week & length held (pts)"] / held["raw (pts)"]
held.rename(index=names).style.format({"raw (pts)": "{:+.1f}", "week & length held (pts)": "{:+.1f}",
                                       "p": "{:.4f}", "left over": "{:.0%}"})

**Check 2: language.** Does the encoder put a unit somewhere because of *what* it says, or because
it is *French*? Subtract from every unit the mean vector of its own language: what all French
units share, and all English units share, is gone. Then k-means again, with the same `K`.

- **Same groups** (adjusted Rand index near the seed-only yardstick): language did not shape them.
- **Other groups:** it did. Then the question is whether French and English *still* differ over
  the new groups, so the shuffle test runs again on them.

This test is strict: if French really holds more love, the French mean holds some love too, and
subtracting it takes part of a real difference away. What survives it, is solid.

In [ ]:
centred = vectors.copy()
for language in settings.COMPARE_LANGUAGES:
    mask = (units["language"] == language).to_numpy()
    centred[mask] -= centred[mask].mean(axis=0)
centred /= np.linalg.norm(centred, axis=1, keepdims=True)

without_language = units.assign(cluster=KMeans(K, n_init=10, random_state=settings.SEED).fit_predict(centred))
language_ari = adjusted_rand_score(units["cluster"], without_language["cluster"])
seed_ari = adjusted_rand_score(units["cluster"], KMeans(K, n_init=10, random_state=settings.SEED + 1).fit_predict(vectors))
print(f"groups with vs. without the language mean: adjusted Rand index {language_ari:.2f} "
      f"(yardstick, only another seed: {seed_ari:.2f})")

new_keywords = cluster_keywords(without_language["message"], without_language["cluster"].to_numpy())
new_names = {k: " / ".join(words[:3]) for k, words in new_keywords.items()}
print("\nwhere the old groups went (rows: old, columns: new)")
display(pd.crosstab(units["cluster"].map(names), without_language["cluster"].map(new_names)))

contrast_without = cluster_contrast(without_language, n_shuffle=N_SHUFFLE)
contrast_without.rename(index=new_names).style.format({
    "share_fr": "{:.1%}", "share_en": "{:.1%}", "difference": "{:+.1%}", "shuffle_mean": "{:+.1%}",
    "beyond_strata": "{:+.1%}", "p": "{:.4f}",
})

**What the two checks mean for my conclusion:** *(how much of each difference survives length and
week? Did language shape the groups? Which differences survive both checks?)*

*Answer:* …

## 6.5.9 The picture for the report

The PCA picture of 6.5.5 again, made to be read: bigger dots, a group palette whose colours do not
run together, and French drawn *on top of* English (there are three times as many English units,
so drawn the other way round they bury the French). The checks and the counts go under it.

In [ ]:
TITLE = "Frans is de taal van de liefde en reacties, Engels van het regelen."
SUBTITLE = "De Franse en Engelse berichten van Louis vallen in andere groepen, ook als lengte en week gelijk zijn"  # draft: check against 6.5.8

order = np.argsort(units["language"].eq("fr").to_numpy(), kind="stable")  # English first, French on top
report = PlotSettings(figsize=(13, 6.2), title=TITLE, xlabel="", ylabel="", max_cols=2,
                      subplot_titles=["wat hij zegt: de groepen van k-means", "in welke taal"])
host = ProjectionPlot(report)
fig, axes = host.create_figure(n_plots=2)
# tab10 without its blue and orange: those are French and English in the right panel
GROUP_COLORS = ["#2ca02c", "#d62728", "#9467bd", "#8c564b", "#bcbd22", "#17becf", "#e377c2", "#7f7f7f"]
dots = {"s": 16, "alpha": 0.7, "linewidth": 0}
host.plot_on_axes(ProjectionPlot(report), axes[0], coordinates=coords[order],
                  labels=units["cluster"].map(names).to_numpy()[order], palette=GROUP_COLORS[:K],
                  hue_order=[names[k] for k in range(K)], **dots)
host.plot_on_axes(ProjectionPlot(report), axes[1], coordinates=coords[order],
                  labels=units["language"].map(LANGUAGE_NAMES).to_numpy()[order],
                  palette={"French": settings.FRENCH_COLOR, "English": settings.ENGLISH_COLOR},
                  hue_order=["French", "English"], **dots)
for ax in axes:
    ax.legend(loc="upper left", markerscale=2, fontsize=9, framealpha=0.85, edgecolor="none")
fig.suptitle(TITLE, fontsize=15, fontweight="bold", y=1.09)
fig.text(0.5, 1.025, SUBTITLE, ha="center", fontsize=11, color="dimgray")

n_lang = units["language"].map(LANGUAGE_NAMES).value_counts()
held_groups = (contrast["p"] < 0.05).sum()
fig.text(0.01, -0.07,
         f"Elke stip is één unit: de berichten van één sessie in één taal, {settings.MIN_UNIT_WORDS}+ woorden, vóór Benji's geboorte. "
         f"{n_lang.get('French', 0):,} Frans, {n_lang.get('English', 0):,} Engels, {units['week'].nunique()} weken.\n"
         f"Assen: de eerste twee PCA-componenten ({cumulative[1]:.0%} van de variatie); de groepen zijn gemaakt in alle "
         f"{vectors.shape[1]} dimensies.\n"
         f"Controle: lengte en week: label geschud binnen week × lengteband ({N_SHUFFLE:,}x): {held_groups} van {K} groepen "
         f"verschillen nog (p < 0.05).\n",
         fontsize=8.5, color="dimgray", va="top")

## 6.5.10 Colour from one route, position from another

**Feedback on 6.5.9 (docent, 2026-10-30):** the k-means groups and the PCA map both come from the
same vectors, so the colours *have* to sit in patches on the map. k-means always returns `K` groups
("why did k-means find three clusters?"), and `K = 6` was never defended. Two routes instead:

1. **Colour, apart from the encoder:** word lists (regex) decide what a unit is about: *love*
   (affection, compliments, encouragement), *planning* (travel, times, cities, house, cleaning,
   interior), *both*, or *other* (grey).
2. **Position, from the encoder:** the same 384-dimensional vectors and PCA map as before.

Now the map can *disagree*: if the love units sit together, the encoder sees the topic; if they are
spread out, it does not. And the French/English comparison runs on labels that no `K` decided.

The lists live in `settings.LOVE_TERMS` and `settings.PLANNING_TERMS`, one regex per term, run on
text without accents (`normalise`), so "chérie", "cherie" and "je pense a toi" all match. Roos's
own French terms come first, then English equivalents, then terms found by counting phrases in
Louis's units: *tu me manques, j'ai hâte de te voir, fais de beaux rêves, ma belle, mon cœur, t'es
la meilleure, trop forte, fier de toi, tu gères, bravo, bisous, faire l'amour / miss you, looking
forward to see you, sweet dreams, you're the best, proud of you, well done, good luck, hugs, kiss*,
and the emoji 😘 🫶 💛 🧡 ❤️ 💪 👏 🍀. Words that looked loving but were not, on reading the
units ("love" in *I love sauna*, "cute" for a dog, "beautiful" for Paris, French *car* = because),
are listed in `settings.py` with the reason they were left out.

**Waarom regex en niet k-means (mijn keuze, 2026-10-30):** ik koos voor regex om ervoor te zorgen
dat de groepen van tevoren al gemaakt werden, zodat ik kan controleren of die groepen als vector bij
elkaar liggen of juist verspreid zijn. K-means vindt namelijk altijd x aantal groepen op basis van de
verkregen data.

k-means (6.5.5–6.5.9) blijft in dit notebook als verkenning, maar gaat niet mee in het verslag.

In [ ]:
units["topic"] = topic_labels(units["message"])
TOPICS = ["love", "planning", "both", "other"]
TOPIC_NAMES = {"love": "liefde & aanmoediging", "planning": "plannen, reizen, huis", "both": "allebei", "other": "overig"}

pd.crosstab(units["topic"], units["language"].map(LANGUAGE_NAMES), normalize="columns").reindex(TOPICS).style.format("{:.1%}")

### Does each term fire in both languages?

A list that is richer in one language measures the list, not Louis. Per term: in how many French
and how many English units it fires. **Add or drop terms in `settings.py`** where a row surprises you.

In [ ]:
for name, terms in settings.UNIT_TOPIC_TERMS.items():
    counts = topic_term_counts(units["message"], units["language"].map(LANGUAGE_NAMES), terms)
    counts.index = [term if len(term) < 70 else term[:67] + "..." for term in counts.index]
    print(f"===== {name}")
    display(counts)

### Read before you believe

Six random units per topic and language. Is a "love" unit really loving, a "planning" unit really
about plans? Count the misses: that is the error rate of the colour route.

In [ ]:
for topic in TOPICS:
    print(f"===== {TOPIC_NAMES[topic]}")
    for language in settings.COMPARE_LANGUAGES:
        rows = units[(units["topic"] == topic) & (units["language"] == language)]
        for text in rows["message"].sample(min(6, len(rows)), random_state=settings.SEED):
            print(f"   {language}  {text[:140]}")

### Where do they lie?

The PCA map of 6.5.5 (the same `coords`) as small multiples: rows are the topics, columns the
languages. Each panel shows *all* units of that language in light grey and only one topic in colour
on top, so no topic hides under another and French does not hide under the three times larger
English. Same axes everywhere: "lower right" is the same place in every panel. Units in *both*
lists stay grey here.

**What the axes go with** is measured, not guessed: the rank correlation (spearman) of each PCA
component with unit length, French, love and planning. The corner notes follow
those signs, so they stay right if PCA flips an axis; the arrow points from the corner into the
cloud. A corner note says where a topic *leans*, not what an axis *is*: the x-axis is mostly length
and language (6.4: the largest direction in the cloud was message length), so the axes get no names.

Under the map: the answer to the question itself (the shuffle test of 6.5.6, label shuffled within
week × length band), the neighbour check in all 384 dimensions, and how the groups were made.

In [ ]:
def result_line(contrast: pd.DataFrame, topic: str) -> str:
    """One sentence per topic for under a figure: the raw shares, and what is left once week and length are held."""
    row = contrast.loc[topic]
    p = "p < 0.001" if row["p"] < 0.001 else f"p = {row['p']:.3f}"
    if row["p"] >= 0.05:
        verdict = "geen verschil dat week en lengte niet al verklaren"
    else:
        verdict = f"vaker in het {'Frans' if row['beyond_strata'] > 0 else 'Engels'}"
    return (f"{TOPIC_NAMES[topic]}: {row['share_fr']:.0%} van zijn Franse units, {row['share_en']:.0%} van zijn "
            f"Engelse ({row['difference'] * 100:+.1f} punten).\n    Met week en lengte gelijk: "
            f"{row['beyond_strata'] * 100:+.1f} punten, {p}: {verdict}.")


# What each PCA axis goes together with (spearman), so the labels follow the data, not a guess.
axis_link = pd.DataFrame({
    measure: [pd.Series(coords[:, i]).corr(pd.Series(values), method="spearman") for i in range(2)]
    for measure, values in {"lengte": units["n_words"], "Frans": units["language"].eq("fr"),
                            "liefde": units["topic"].eq("love"), "planning": units["topic"].eq("planning")}.items()
}, index=["x", "y"])
corner_notes = {
    topic: (f"{'upper' if axis_link.loc['y', name] > 0 else 'lower'} {'right' if axis_link.loc['x', name] > 0 else 'left'}",
            text)
    for topic, name, text in [("love", "liefde", "liefdesberichtjes"), ("planning", "planning", "planningsberichtjes")]
}
display(axis_link.style.format("{:+.2f}"))


def topic_map(mask: np.ndarray, title: str, contrast: pd.DataFrame):
    """The 2 x 2 map for the units in `mask`, with the shuffle test, the neighbour check and the method under it."""
    weeks = units.loc[mask, "week"]
    maps = PlotSettings(figsize=(12, 9.5), title=title, xlabel="", ylabel="", max_cols=2)
    fig, axes = TopicMapPanels(maps).plot(n_plots=4, coordinates=coords[mask],
                                          topics=units.loc[mask, "topic"].to_numpy(),
                                          languages=units.loc[mask, "language"].to_numpy(),
                                          names=TOPIC_NAMES, pair_labels={"fr": "Frans", "en": "Engels"},
                                          extent=coords,  # same axes for every subset
                                          notes=corner_notes)

    near = neighbour_agreement(vectors[mask], units.loc[mask, "topic"].to_numpy())
    neighbours = "\n".join(
        f"    {TOPIC_NAMES[t]}: {near.loc[t, 'neighbours_same']:.0%} van de 15 dichtstbijzijnde units ook, "
        f"bij toeval {near.loc[t, 'base_rate']:.0%} ({near.loc[t, 'lift']:.1f}x zo vaak)" for t in ["love", "planning"])
    fig.text(0.01, -0.01, "\n".join(result_line(contrast, t) for t in ["love", "planning"]),
             fontsize=11, va="top", fontweight="bold")
    fig.text(0.01, -0.095,
             f"Burencheck in alle {vectors.shape[1]} dimensies: liggen units met hetzelfde onderwerp bij elkaar?\n"
             f"{neighbours}\n"
             "    De encoder legt ze dus bij elkaar, zonder de woordenlijsten te kennen.",
             fontsize=10, va="top")
    fig.text(0.01, -0.185,
             "Groepen: gemaakt met een regex-patroon (woordenlijsten in het Frans en Engels), vooraf en los van de encoder; "
             "units met termen uit beide lijsten staan grijs.\n"
             f"Assen: de eerste twee PCA-componenten ({cumulative[1]:.0%} van de variatie). x hangt samen met lengte "
             f"({axis_link.loc['x', 'lengte']:+.2f}) en Frans ({axis_link.loc['x', 'Frans']:+.2f}); y met planning "
             f"({axis_link.loc['y', 'planning']:+.2f}) en liefde ({axis_link.loc['y', 'liefde']:+.2f}), spearman.\n"
             f"Units van {settings.MIN_UNIT_WORDS}+ woorden, weken van {weeks.min():%d %b %Y} tot {weeks.max():%d %b %Y}. "
             f"p: taallabel {N_SHUFFLE:,}x geschud binnen week × lengteband.",
             fontsize=8.5, color="dimgray", va="top")
    return fig, axes


n_lang = units["language"].map(LANGUAGE_NAMES).value_counts()
topic_contrast = cluster_contrast(units, cluster="topic", n_shuffle=N_SHUFFLE).reindex(TOPICS)
_ = topic_map(np.ones(len(units), dtype=bool), "Waar praat Louis over liefde, en waar over plannen? (vóór Benji)",
              topic_contrast)

The map shows 10% of the variance. Whether the encoder puts same-topic units together *in all 384
dimensions*: per topic, the share of each unit's 15 nearest neighbours with the same topic, against
that topic's share overall (`lift` 1 = no better than chance).

In [ ]:
neighbour_agreement(vectors, units["topic"].to_numpy()).reindex(TOPICS).rename(index=TOPIC_NAMES).style.format(
    {"neighbours_same": "{:.1%}", "base_rate": "{:.1%}", "lift": "{:.2f}"})

### French against English, on the word-list topics

The question itself, now on labels no `K` decided: per topic the share of Louis's French and of his
English units, with the shuffle test of 6.5.6 (label shuffled **within week × length band**). That
band matters more here than before: a long unit has more words, so more chance that *some* term
fires, and English units are the long ones.

In [ ]:
dumbbell = PlotSettings(figsize=(9, 3.6), title="TITLE: write your own claim here, after reading the numbers",
                        xlabel="aandeel van zijn units in die taal (%)", ylabel="")
fig, ax = LanguageDumbbellPlot(dumbbell).plot(contrast=topic_contrast, names=TOPIC_NAMES,
                                              pair_labels={"fr": "Frans", "en": "Engels"})
fig.text(0.01, -0.06, f"Units van {settings.MIN_UNIT_WORDS}+ woorden vóór Benji's geboorte: "
         f"{n_lang.get('French', 0):,} Frans, {n_lang.get('English', 0):,} Engels, {units['week'].nunique()} weken. "
         f"Onderwerp: woordenlijsten. p: label geschud binnen week × lengteband, {N_SHUFFLE:,}x.",
         fontsize=8, color="dimgray")

topic_contrast.rename(index=TOPIC_NAMES).style.format({
    "share_fr": "{:.1%}", "share_en": "{:.1%}", "difference": "{:+.1%}", "shuffle_mean": "{:+.1%}",
    "beyond_strata": "{:+.1%}", "p": "{:.4f}",
})

### Before and after the move

The same map twice: the long-distance months up to the move to Utrecht (`settings.MOVE_DATE`),
and the months from the move to Benji's birth. Same `coords`, so a place on one map is the same
place on the other; only the units differ. The shuffle test runs per period. A unit belongs to the
week its first message falls in, so the week of 28 Oct 2024 (with the 1st of November in it) still
counts as "before".

After the move there are far fewer units (Louis and Roos talk in person), so the p-values there
rest on less data.

In [ ]:
PERIODS = {
    "tot de verhuizing naar Utrecht": (units["week"] < settings.MOVE_DATE).to_numpy(),
    "van de verhuizing tot Benji": (units["week"] >= settings.MOVE_DATE).to_numpy(),
}
period_contrast = {}
for period, mask in PERIODS.items():
    period_contrast[period] = cluster_contrast(units[mask], cluster="topic", n_shuffle=N_SHUFFLE).reindex(TOPICS)
    _ = topic_map(mask, f"Liefde en plannen, {period}", period_contrast[period])

pd.concat(period_contrast, names=["period"]).rename(index=TOPIC_NAMES, level="topic").style.format({
    "share_fr": "{:.1%}", "share_en": "{:.1%}", "difference": "{:+.1%}", "shuffle_mean": "{:+.1%}",
    "beyond_strata": "{:+.1%}", "p": "{:.4f}",
})

### Do the corners hold when the length is the same?

French units are shorter, and the x-axis is mostly length. So "love lies lower right" could be
"short units lie right". The check: within one language **and** one length band, where do the love
(planning) units sit, against the other units of that same band? If a topic still sits lower (higher)
in every band, that part of the corner is the topic, not the length. Long-distance months only.

In [ ]:
LONG_DISTANCE = PERIODS["tot de verhuizing naar Utrecht"]
placed = units[LONG_DISTANCE].assign(x=coords[LONG_DISTANCE, 0], y=coords[LONG_DISTANCE, 1])
corner_check = pd.concat({
    TOPIC_NAMES[topic]: placed.assign(group=placed["topic"].eq(topic).map({True: "topic", False: "rest"}))
    .groupby(["language", "length_band", "group"], observed=True)[["x", "y"]].median()
    .unstack("group")
    .pipe(lambda d: pd.DataFrame({"x: topic - rest": d[("x", "topic")] - d[("x", "rest")],
                                  "y: topic - rest": d[("y", "topic")] - d[("y", "rest")]}))
    for topic in ["love", "planning"]
}, axis=1)
corner_check.rename(index=LANGUAGE_NAMES, level="language").style.format("{:+.2f}").background_gradient(
    cmap="PiYG", vmin=-0.2, vmax=0.2)

## 6.5.11 Each language's own cloud, during the long-distance months

Until now one PCA was fitted on French and English together, so its first direction was largely
*which language, how long*. Here PCA is fitted on **one language at a time**: the largest spreads
*within* his French, and *within* his English. Top row coloured by topic (word lists), bottom row by
length band. In the titles: the share of the variance the two components hold, and how many
components 90% needs.

The axes of each panel are that language's own: a place in the French panel says nothing about the
same place in the English one. Under the figure, the rank correlation of each axis with length,
love and planning, so the axes are read from numbers, not from the eye.

In [ ]:
own_clouds, own_axes = {}, {}
for language in settings.COMPARE_LANGUAGES:
    mask = LONG_DISTANCE & (units["language"] == language).to_numpy()
    own_pca = PCA().fit(vectors[mask])
    own_clouds[language] = (mask, own_pca.transform(vectors[mask])[:, :2], own_pca.explained_variance_ratio_)

titles = []
for colour_by in ["onderwerp", "lengte"]:
    for language, (mask, xy, ratio) in own_clouds.items():
        titles.append(f"{'Frans' if language == 'fr' else 'Engels'} ({mask.sum():,} units), kleur: {colour_by}\n"
                      f"2 componenten: {ratio[:2].sum():.0%} van de variatie; 90% vraagt er "
                      f"{np.searchsorted(np.cumsum(ratio), 0.9) + 1}")
clouds = PlotSettings(figsize=(13, 11), title="De wolk van elke taal apart, tot de verhuizing (eigen PCA per taal)",
                      xlabel="", ylabel="", max_cols=2, subplot_titles=titles)
host = ProjectionPlot(clouds)
fig, axes = host.create_figure(n_plots=4)
bands = list(units["length_band"].cat.categories)
for i, (language, (mask, xy, ratio)) in enumerate(own_clouds.items()):
    rows = units[mask]
    order = np.argsort(rows["topic"].ne("other").to_numpy(), kind="stable")  # grey first
    host.plot_on_axes(ProjectionPlot(clouds), axes[i], coordinates=xy[order],
                      labels=rows["topic"].map(TOPIC_NAMES).to_numpy()[order],
                      palette={TOPIC_NAMES[t]: settings.TOPIC_COLORS[t] for t in TOPICS},
                      hue_order=[TOPIC_NAMES[t] for t in TOPICS], s=14, alpha=0.8, linewidth=0)
    host.plot_on_axes(ProjectionPlot(clouds), axes[i + 2], coordinates=xy,
                      labels=rows["length_band"].astype(str).to_numpy(), palette="viridis",
                      hue_order=bands, s=14, alpha=0.8, linewidth=0)
    own_axes[LANGUAGE_NAMES[language]] = pd.DataFrame({
        measure: [pd.Series(xy[:, k]).corr(pd.Series(values), method="spearman") for k in range(2)]
        for measure, values in {"lengte": rows["n_words"].to_numpy(), "liefde": rows["topic"].eq("love").to_numpy(),
                                "planning": rows["topic"].eq("planning").to_numpy()}.items()
    }, index=["x", "y"])
for ax in axes:
    ax.legend(loc="upper right", markerscale=1.5, fontsize=8, framealpha=0.85, edgecolor="none",
              title="woorden" if ax in axes[2:] else None, title_fontsize=8)
fig.text(0.01, -0.01, "Elke stip is een unit van Louis (4+ woorden) tijdens de lange-afstandsrelatie. Kleur onderwerp: "
         "woordenlijsten (regex). Elke taal heeft zijn eigen PCA, dus de assen verschillen per kolom.",
         fontsize=8.5, color="dimgray", va="top")

similar = {LANGUAGE_NAMES[lang]: np.median((vectors[m] @ vectors[m].T)[np.triu_indices(m.sum(), 1)])
           for lang, (m, _, _) in own_clouds.items()}
print("how alike two random units of one language are (median cosine): "
      + ", ".join(f"{lang} {value:.2f}" for lang, value in similar.items()))
pd.concat(own_axes, names=["language", "axis"]).style.format("{:+.2f}").background_gradient(
    cmap="PiYG", vmin=-0.5, vmax=0.5)

**What the two clouds say:** *(in my own words)*

1. **Shape:** one cloud or separate clumps? Which language's cloud is more spread out, and which holds
   more in two components?
2. **Which direction is largest within each language?** Read the table: does length, love or
   planning go with the x- or y-axis in French, and in English?
3. **Similarity:** French units are more alike each other (median cosine). Is that because Louis
   says similar things in French, or because short texts ("bonne nuit 😘") look alike anyway?

*Answer:* …

## 6.5.12 The picture for the report (replaces 6.5.9)

One message: a larger share of Louis's French than of his English is loving or encouraging. So two
panels, French and English, on the same map and the same axes:

- **love** large and on top, **planning** small and faded (context: where the other topic lies),
  all other units light grey;
- the **share** of love as a big number in each panel: the comparison the eye should make;
- the subtitle carries the test (length and week held), the footnote how the groups were made and
  the neighbour check, which is what the map itself shows: the word-list groups lie together.

Long-distance months only (`LONG_DISTANCE`): after the move the love difference does not hold (6.5.10).
No axis names: the x-axis is mostly length and language, not a topic.

In [ ]:
love = period_contrast["tot de verhuizing naar Utrecht"].loc["love"]
p = "p < 0.001" if love["p"] < 0.001 else f"p = {love['p']:.3f}"
TITLE = "Frans is de taal van de liefde"
SUBTITLE = (f"Tijdens de lange-afstandsrelatie is {love['share_fr']:.0%} van Louis' Franse berichtjes liefdevol of "
            f"aanmoedigend, tegen {love['share_en']:.0%} van zijn Engelse.\nOok bij gelijke lengte en week blijft "
            f"{love['beyond_strata'] * 100:+.0f} procentpunt verschil over ({p}).")

near = neighbour_agreement(vectors[LONG_DISTANCE], units.loc[LONG_DISTANCE, "topic"].to_numpy())
weeks = units.loc[LONG_DISTANCE, "week"]
report_map = PlotSettings(figsize=(13, 6.2), title="", xlabel="", ylabel="", max_cols=2)
fig, axes = ReportMapPanels(report_map).plot(n_plots=2, coordinates=coords[LONG_DISTANCE],
                                             topics=units.loc[LONG_DISTANCE, "topic"].to_numpy(),
                                             languages=units.loc[LONG_DISTANCE, "language"].to_numpy(),
                                             pair_labels={"fr": "Frans", "en": "Engels"},
                                             extent=coords, notes=corner_notes)
fig.suptitle(TITLE, fontsize=19, fontweight="bold", y=1.13)
fig.text(0.5, 1.02, SUBTITLE, ha="center", fontsize=12, color="dimgray")
fig.legend(handles=[Line2D([], [], marker="o", linestyle="", markersize=8, color=settings.TOPIC_COLORS[t],
                               alpha=a, label=TOPIC_NAMES[t]) for t, a in [("love", 0.9), ("planning", 0.45), ("other", 1)]],
           loc="lower center", bbox_to_anchor=(0.5, -0.07), ncol=3, frameon=False, fontsize=10)
fig.text(0.01, -0.1,
         "Groepen: vooraf gemaakt met een regex-patroon (woordenlijsten in het Frans en Engels), los van de encoder. "
         f"Burencheck in alle {vectors.shape[1]} dimensies: van de 15 dichtstbijzijnde\nunits van een liefdesberichtje "
         f"is {near.loc['love', 'neighbours_same']:.0%} ook liefdevol, bij toeval {near.loc['love', 'base_rate']:.0%}: "
         "de encoder legt ze bij elkaar zonder de woordenlijsten te kennen. Positie: de eerste twee PCA-componenten "
         f"({cumulative[1]:.0%} van de variatie).\nUnits van {settings.MIN_UNIT_WORDS}+ woorden (berichten van één sessie "
         f"in één taal), {weeks.min():%d %b %Y} tot {weeks.max():%d %b %Y}. "
         f"p: taallabel {N_SHUFFLE:,}x geschud binnen week × lengteband.",
         fontsize=8.5, color="dimgray", va="top")

**What I see:** *(in my own words)*

1. **The map and the neighbours:** do the love and planning units sit together, in the picture and
   in 384 dimensions? What does that say about the encoder, and about the k-means groups of 6.5.5?
2. **The term counts:** is one language's list richer than the other's? Which terms did I add or drop?
3. **French vs. English:** how much of the love and planning difference is left once week and
   length are held (`beyond_strata`)? Does that fit my expectation?
4. **Before and after the move:** does the difference hold in both periods, or does one period carry it?
5. **Which picture goes in the report**, and what is its title?

*Answer:* …